In [2]:
import os
import yaml
import pandas as pd
from collections import Counter
from ultralytics import YOLO

DATASET = r"C:\Users\Localws\Bottle-Detection-1"
DATA_YAML = os.path.join(DATASET, "data.yaml")
V1_MODEL = os.path.join(DATASET, "best.pt")

with open(DATA_YAML, "r") as file:
    data = yaml.safe_load(file)

class_names = data["names"]

print("Classes:", class_names)

Classes: ['bottle']


In [3]:
train_images_path = os.path.join(DATASET, "train", "images")
train_labels_path = os.path.join(DATASET, "train", "labels")

class_image_counts = Counter()

image_files = [
    f for f in os.listdir(train_images_path)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

for label_file in os.listdir(train_labels_path):
    if not label_file.endswith(".txt"):
        continue

    label_path = os.path.join(train_labels_path, label_file)
    classes_in_image = set()

    with open(label_path, "r") as file:
        for line in file:
            parts = line.strip().split()

            if len(parts) >= 5:
                class_id = int(float(parts[0]))
                classes_in_image.add(class_id)

    for class_id in classes_in_image:
        class_image_counts[class_id] += 1

print("Total training images:", len(image_files))
print("\nImages containing each class:")

for class_id, name in enumerate(class_names):
    print(f"{class_id} - {name}: {class_image_counts[class_id]} images")

Total training images: 139

Images containing each class:
0 - bottle: 139 images


In [4]:
max_samples = max(class_image_counts.values())

print("Classes with fewer samples:")

for class_id, name in enumerate(class_names):
    count = class_image_counts[class_id]

    if count < max_samples:
        print(f"{name}: {count} images")

Classes with fewer samples:


In [5]:
from collections import Counter
import os

valid_labels_path = os.path.join(DATASET, "valid", "labels")

valid_class_counts = Counter()

for label_file in os.listdir(valid_labels_path):
    if not label_file.endswith(".txt"):
        continue

    label_path = os.path.join(valid_labels_path, label_file)

    with open(label_path, "r") as file:
        for line in file:
            parts = line.strip().split()

            if len(parts) >= 5:
                class_id = int(float(parts[0]))
                valid_class_counts[class_id] += 1

print("Validation class IDs:")
print(valid_class_counts)

print("\nClasses in data.yaml:")
for i, name in enumerate(class_names):
    print(f"{i} -> {name}")

Validation class IDs:
Counter({0: 88})

Classes in data.yaml:
0 -> bottle


In [6]:
train_class_ids = set()

for label_file in os.listdir(train_labels_path):
    if not label_file.endswith(".txt"):
        continue

    label_path = os.path.join(train_labels_path, label_file)

    with open(label_path, "r") as file:
        for line in file:
            parts = line.strip().split()

            if len(parts) >= 5:
                train_class_ids.add(int(float(parts[0])))

print("Training class IDs:", sorted(train_class_ids))
print("Validation class IDs:", sorted(valid_class_counts.keys()))

Training class IDs: [0]
Validation class IDs: [0]


In [7]:
model_v1 = YOLO(V1_MODEL)

print("V1 model classes:")
print(model_v1.names)

print("\nNumber of V1 classes:")
print(len(model_v1.names))

V1 model classes:
{0: 'bottle'}

Number of V1 classes:
1


In [8]:
import shutil
import os

backup_path = r"C:\Users\Localws\Bottle-Detection-1_backup"

if not os.path.exists(backup_path):
    shutil.copytree(DATASET, backup_path)
    print("Backup created successfully.")
else:
    print("Backup already exists.")

Backup already exists.


In [9]:
for split in ["train", "valid", "test"]:
    labels_path = os.path.join(DATASET, split, "labels")

    for label_file in os.listdir(labels_path):
        if not label_file.endswith(".txt"):
            continue

        label_path = os.path.join(labels_path, label_file)

        new_lines = []

        with open(label_path, "r") as file:
            for line in file:
                parts = line.strip().split()

                if len(parts) >= 5:
                    parts[0] = "0"
                    new_lines.append(" ".join(parts))

        with open(label_path, "w") as file:
            file.write("\n".join(new_lines))

print("All labels converted to class 0.")

All labels converted to class 0.


In [10]:
data["nc"] = 1
data["names"] = ["bottle"]

with open(DATA_YAML, "w") as file:
    yaml.safe_dump(data, file, sort_keys=False)

print("data.yaml updated successfully.")
print(data)

data.yaml updated successfully.
{'names': ['bottle'], 'nc': 1, 'roboflow': {'license': 'CC BY 4.0', 'project': 'bottle-detection-7gu4t', 'url': 'https://universe.roboflow.com/eman-fatima-e1ncy/bottle-detection-7gu4t/dataset/1', 'version': 1, 'workspace': 'eman-fatima-e1ncy'}, 'test': '../test/images', 'train': '../train/images', 'val': '../valid/images'}


In [11]:
for split in ["train", "valid", "test"]:
    labels_path = os.path.join(DATASET, split, "labels")
    class_ids = set()

    for label_file in os.listdir(labels_path):
        if not label_file.endswith(".txt"):
            continue

        with open(os.path.join(labels_path, label_file), "r") as file:
            for line in file:
                parts = line.strip().split()

                if len(parts) >= 5:
                    class_ids.add(int(parts[0]))

    print(f"{split}: {sorted(class_ids)}")

train: [0]
valid: [0]
test: [0]


In [12]:
model_v1 = YOLO(V1_MODEL)

v1_results = model_v1.val(
    data=DATA_YAML,
    split="val"
)

v1_precision = v1_results.box.mp
v1_recall = v1_results.box.mr
v1_map50 = v1_results.box.map50
v1_map5095 = v1_results.box.map

print("V1 Results")
print(f"Precision: {v1_precision:.4f}")
print(f"Recall: {v1_recall:.4f}")
print(f"mAP@50: {v1_map50:.4f}")
print(f"mAP@50-95: {v1_map5095:.4f}")

Ultralytics 8.4.115  Python-3.13.7 torch-2.13.0+cpu CPU (Intel Core i5-5300U 2.30GHz)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.70.3 ms, read: 255.2103.5 MB/s, size: 1470.5 KB)
val: Scanning C:\Users\Localws\Bottle-Detection-1\valid\labels.cache... 39 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 39/39 1.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 14.2s/it 42.6s31.6s
                   all         39         88      0.777      0.511      0.576      0.326
Speed: 2.7ms preprocess, 191.3ms inference, 0.0ms loss, 4.5ms postprocess per image
Results saved to C:\Users\Localws\runs\detect\val-20
V1 Results
Precision: 0.7772
Recall: 0.5114
mAP@50: 0.5764
mAP@50-95: 0.3263


In [13]:
from ultralytics import YOLO

model_v2 = YOLO(V1_MODEL)

model_v2.train(
    data=DATA_YAML,
    epochs=4,
    imgsz=320,
    batch=4,
    workers=0,

    # Data augmentation
    degrees=10,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    mosaic=0.5,

    project=r"C:\Users\Localws\Bottle-Detection-1\Day37_Training",
    name="V2"
)

New https://pypi.org/project/ultralytics/8.4.127 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.115  Python-3.13.7 torch-2.13.0+cpu CPU (Intel Core i5-5300U 2.30GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\Localws\Bottle-Detection-1\data.yaml, degrees=10, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=4, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.5, hsv_v=0.3, imgsz=320, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002AD813BDEB0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [14]:
V2_MODEL = r"C:\Users\Localws\Bottle-Detection-1\Day37_Training\V2\weights\best.pt"

import os

print("V2 model exists:", os.path.exists(V2_MODEL))

V2 model exists: True


In [15]:
model_v2 = YOLO(V2_MODEL)

v2_results = model_v2.val(
    data=DATA_YAML,
    split="val"
)

v2_precision = v2_results.box.mp
v2_recall = v2_results.box.mr
v2_map50 = v2_results.box.map50
v2_map5095 = v2_results.box.map

print("V2 Results")
print(f"Precision: {v2_precision:.4f}")
print(f"Recall: {v2_recall:.4f}")
print(f"mAP@50: {v2_map50:.4f}")
print(f"mAP@50-95: {v2_map5095:.4f}")

Ultralytics 8.4.115  Python-3.13.7 torch-2.13.0+cpu CPU (Intel Core i5-5300U 2.30GHz)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.50.1 ms, read: 319.198.7 MB/s, size: 973.6 KB)
val: Scanning C:\Users\Localws\Bottle-Detection-1\valid\labels.cache... 39 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 39/39 4.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 18.2s/it 54.7s42.1s
                   all         39         88      0.776      0.489      0.581      0.277
Speed: 4.0ms preprocess, 238.7ms inference, 0.0ms loss, 5.1ms postprocess per image
Results saved to C:\Users\Localws\runs\detect\val-21
V2 Results
Precision: 0.7756
Recall: 0.4886
mAP@50: 0.5809
mAP@50-95: 0.2766


In [16]:
comparison = pd.DataFrame({
    "Metric": ["Precision", "Recall", "mAP@50", "mAP@50-95"],
    "V1": [0.7772, 0.5114, 0.5764, 0.3263],
    "V2": [0.7756, 0.4886, 0.5809, 0.2766]
})

comparison["Change"] = comparison["V2"] - comparison["V1"]

comparison

,Metric,V1,V2,Change
0,Precision,0.7772,0.7756,-0.0016
1,Recall,0.5114,0.4886,-0.0228
2,mAP@50,0.5764,0.5809,0.0045
3,mAP@50-95,0.3263,0.2766,-0.0497
